# Chapter 10 — An all-sky search with Gaia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/10_gaia_allsky.ipynb)

**Goal.** Gaia observes the whole sky, uniformly enough to search all of it at once for compact stellar overdensities.
Koposov, Belokurov & Torrealba (2017, MNRAS 470, 2702, https://ui.adsabs.harvard.edu/abs/2017MNRAS.470.2702K) did this
with Gaia DR1 — positions and G magnitudes only — and discovered two new star clusters, Gaia 1 and Gaia 2. We reproduce
their all-sky significance map (their Figure 1) with DR1, then repeat it with DR3, and with DR3 plus cuts that DR1 could
not make: removing nearby stars with measured parallaxes, and a colour cut.

**You will learn**
1. to count ~10⁹ stars on a HEALPix grid *inside the database*, downloading only the counts;
2. to smooth a HEALPix map with a Gaussian kernel, and to compute a local mean and variance in an annulus, using
   spherical harmonics;
3. why several kernel sizes are needed (Gaia 2 is 2′ across, Crater 2 is 30′);
4. to compare data sets by the known objects they recover and the spurious peaks they produce.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('healpy') is None:                  # Colab does not ship healpy
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'healpy'], check=True)
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp                                             # HEALPix grids, sky plots and spherical harmonics
import re                                                       # to read the cone centres from the manifest
from scipy.ndimage import gaussian_filter                       # smoothing of the report-card maps

# --- Data access (identical in every chapter) -------------------------------------------------------------
# The data files are small extracts of survey catalogues, attached to a GitHub release of the tutorial.
# MANIFEST.json lists, for each file, the SQL that made it and its SHA-256 checksum.
DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release (again if the local copy is stale), check its SHA-256."""
    path = cache / f'{name}.parquet'
    if not path.exists():                          # first use: fetch the file into ./data
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MANIFEST[name]['sha256']:   # stale or damaged copy
        print(f'downloading {path.name} again (checksum mismatch) ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    # the checksum guarantees that everyone runs this chapter on exactly the same rows
    assert hashlib.sha256(path.read_bytes()).hexdigest() == MANIFEST[name]['sha256'], f'{path} does not match the manifest'
    return pd.read_parquet(path)

## 10.1 Counting a billion stars without downloading them

Gaia DR1 has 1.1 × 10⁹ sources, DR3 1.8 × 10⁹. We need only how many stars fall in each pixel of a sky grid. A Gaia
`source_id` encodes the source's position: `source_id // 2**35` is its index in the nested HEALPix grid of level 12
(nside = 4096). Dividing by 2³⁹ instead gives level 10 (nside = 1024, pixels of 3.4′). So the database can build the
histogram with a `GROUP BY` and return ~12 million counts instead of ~10⁹ rows:

In [ ]:
# The SQL that built the two count tables, as stored in the manifest. source_id / 549755813888 (= 2**39) is the index of
# the star's pixel in the NESTED HEALPix grid of level 10 (nside = 1024). The {lo}, {hi} placeholders: the query is run
# 3072 times, on source_id ranges of 2**51 each, i.e. once per pixel of the level-4 HEALPix grid (12 * 4**4 = 3072).
print(MANIFEST['gaia_dr1_counts']['queries'][0])
print()
print(MANIFEST['gaia_dr3_counts']['queries'][0])
print()
print(MANIFEST['gaia_dr3_counts_plxcut']['queries'][0])

The selection follows Koposov et al. (2017): $17 < G < 21$ (brighter stars are mostly nearby foreground), and
`astrometric_excess_noise` below $10^{0.15(G - 15) + 0.25}$ mas, which removes most galaxies (an extended source scatters
Gaia's one-dimensional position measurements). From DR3 we use `n_k17`, the same selection as in DR1, and a second
table, `gaia_dr3_counts_plxcut`, with four counts per pixel (the SQL is printed above):

- `n_plx1`: without stars of parallax ϖ > 1 mas; `n_plx02`: without stars of ϖ > 0.2 mas. Only the measured parallax
  is used, not its uncertainty; stars without a parallax are kept (Section 10.5 discusses what these cuts do);
- `n_plx1_cmd`, `n_plx02_cmd`: also $0.3 < G_{\rm BP} - G_{\rm RP} < 1.3$, the colours of old, metal-poor turn-off and
  giant stars (stars without a colour are dropped).

(The DR3 count file also holds `n_far` and `n_far_cmd`, an earlier version of the parallax cut that used the
uncertainty, $\varpi - 3\sigma_\varpi > 1$ mas; they are not used.)

In [ ]:
NSIDE = 1024                    # HEALPix level 10, the grid of the count tables: pixels of 3.4' (12 * 1024**2 pixels)
NPIX = hp.nside2npix(NSIDE)     # 12,582,912


def to_map(df, col):
    """Full-sky RING-ordered map of a count column (the files are in NESTED order, which healpy's smoothing does not use).

    Pixels missing from the table (no star passed the cuts) are set to zero. Returns a float array of shape (NPIX,).
    """
    m = np.zeros(NPIX)
    m[df.hpx.to_numpy()] = df[col].to_numpy()       # hpx: NESTED pixel index computed in the SQL
    return hp.reorder(m, n2r=True)


dr1 = load('gaia_dr1_counts')                       # one row per non-empty pixel: columns hpx, n_k17
dr3 = load('gaia_dr3_counts')                       # columns hpx, n_k17 (its n_far, n_far_cmd are not used here)
cut = load('gaia_dr3_counts_plxcut')                # columns hpx, n_plx1, n_plx1_cmd, n_plx02, n_plx02_cmd
# the K17 selection of DR1 and DR3, and DR3 without stars of parallax > 1 mas or > 0.2 mas (stars without a parallax are
# kept), each also with 0.3 < BP-RP < 1.3
maps = {'DR1': to_map(dr1, 'n_k17'), 'DR3': to_map(dr3, 'n_k17'),
        'DR3, ϖ ≤ 1': to_map(cut, 'n_plx1'), 'DR3, ϖ ≤ 1 + colour': to_map(cut, 'n_plx1_cmd'),
        'DR3, ϖ ≤ 0.2': to_map(cut, 'n_plx02'), 'DR3, ϖ ≤ 0.2 + colour': to_map(cut, 'n_plx02_cmd')}
del dr1, dr3, cut                                        # the tables are no longer needed; free ~12M-row data frames
for k, m in maps.items():
    print(f'{k:18s}: {m.sum():.3e} stars')

# log10 of the counts per pixel, grey scale from 10**0.5 ~ 3 to 10**3 stars per pixel; empty pixels are set to 1 count
# (log10 = 0) so that the logarithm is defined. Equatorial coordinates, healpy's default Mollweide view.
fig = plt.figure(figsize=(14, 4))
hp.mollview(np.log10(np.maximum(maps['DR1'], 1)), fig=fig, sub=(1, 2, 1), title='Gaia DR1, 17 < G < 21: log10 counts per pixel',
            cmap='Greys', min=0.5, max=3)
hp.mollview(np.log10(np.maximum(maps['DR3'], 1)), fig=fig, sub=(1, 2, 2), title='Gaia DR3, same selection', cmap='Greys', min=0.5, max=3)

**Figure 10.1.** Gaia star counts per HEALPix pixel (nside = 1024, 3.4′ pixels) for $17 < G < 21$ and
`astrometric_excess_noise` below $10^{0.15(G-15)+0.25}$ mas, in equatorial coordinates (Mollweide projection centred on
RA = 0°, RA increasing to the left). Grey scale: $\log_{10}$ of the counts per pixel, from $10^{0.5} \approx 3$ (white) to
$10^3$ (black); empty pixels are shown as 1 count. Left: Gaia DR1 ($9.70 \times 10^8$ stars in total); right: DR3 with
the same selection ($1.56 \times 10^9$ stars). The Galactic disc and bulge form the dark arc; the Magellanic Clouds are the
two dark blobs at the bottom, left of centre. In DR1, the stripes and arcs are Gaia's scanning pattern: after only 14
months, some areas had been observed many more times than others, so the completeness at G ≈ 20–21 varies across the
sky. DR3 (34 months) is far more uniform.

## 10.2 Kernel density and significance on the sphere

Following Koposov et al. (2017), the density is the histogram $H$ convolved with a normalised Gaussian kernel,
$D = K * H$, and the significance is

$$ S = \frac{D - \langle D\rangle_A}{\sqrt{{\rm Var}_A(D)}}, $$

where the mean and variance are taken over an annulus $A$ around each point. Using the local variance rather than a
Poisson estimate absorbs the extra fluctuations of the scanning pattern. The paper does not give the annulus radii; we
use 3σ–6σ of the kernel.

One numerical safeguard: ${\rm Var}_A(D) = \langle D^2\rangle_A - \langle D\rangle_A^2$ is a difference of two smoothed
maps and can cancel to ~0 (or below) through numerical ringing when the kernel is not much larger than a pixel. In a first
version this produced S ~ 10⁵ in patches of the sky. We therefore never let the variance fall below the Poisson variance
of $D$, $\langle D\rangle_A\,\Omega_{\rm pix}/(4\pi\sigma^2)$, and use kernels of at least 4′ (pixels are 3.4′).

On the sphere, convolution with a function of angular distance is a multiplication in spherical-harmonic space:
`healpy.smoothing` does it for a Gaussian, and for the annulus we compute its harmonic coefficients from its radial
profile (`beam2bl`), normalised so that it averages.

In [ ]:
LMAX = 3 * NSIDE - 1                            # highest multipole of the harmonic transforms (healpy's default)
_theta = np.linspace(0, np.radians(5), 5000)    # radial grid for the annulus profile [rad]: 0-5 deg in 0.06' steps;
                                                # the largest annulus used (6 x 12') ends at 1.2 deg


def annulus_bl(r1_deg, r2_deg):
    """Harmonic coefficients of a normalised top-hat annulus r1 < theta < r2.

    Radii in degrees. bl[0] is the integral of the profile, so after dividing by it, multiplying a map's harmonic
    coefficients by bl gives at each point the *mean* of the map over the annulus around it.
    """
    # radial profile: 1 inside the annulus, 0 outside
    b = ((_theta > np.radians(r1_deg)) & (_theta < np.radians(r2_deg))).astype(float)
    bl = hp.beam2bl(b, _theta, LMAX)                                                     # radial profile -> b_l
    return bl / bl[0]


OMEGA_PIX = 4 * np.pi / NPIX                    # solid angle of one pixel [sr]


def significance(H, sigma_arcmin):
    """S map for Gaussian kernel sigma and annulus 3-6 sigma (RING maps); variance floored at the Poisson value.

    H: counts per pixel (RING order, shape (NPIX,)); sigma_arcmin: kernel width [arcmin].
    Returns S = (D - <D>_A) / sqrt(Var_A(D)) as float32 (half the memory of float64 for a 12.6M-pixel map).
    """
    sig = np.radians(sigma_arcmin / 60)                                   # kernel sigma [rad]
    # D = K * H: counts smoothed with a normalised Gaussian (iter=1: one refinement step of the harmonic transform
    # instead of healpy's default 3 -- faster, slightly less accurate)
    D = hp.smoothing(H, sigma=sig, iter=1)
    bl = annulus_bl(3 * sigma_arcmin / 60, 6 * sigma_arcmin / 60)        # annulus from 3 sigma to 6 sigma [deg]
    mean = hp.alm2map(hp.almxfl(hp.map2alm(D, lmax=LMAX, iter=1), bl), NSIDE)      # <D>_A
    msq = hp.alm2map(hp.almxfl(hp.map2alm(D**2, lmax=LMAX, iter=1), bl), NSIDE)    # <D^2>_A
    # Poisson variance of D: smoothing Poisson counts with a normalised Gaussian of width sig gives
    # Var(D) = <D> * Omega_pix / (4 pi sig^2). Used as a floor where <D^2>_A - <D>_A^2 cancels to ~0 or below.
    var_poisson = np.maximum(mean, 0) * OMEGA_PIX / (4 * np.pi * sig**2)
    # the 1e-12 only prevents a division by zero where both variances vanish (pixels with no stars around them)
    return ((D - mean) / np.sqrt(np.maximum(msq - mean**2, var_poisson) + 1e-12)).astype(np.float32)


S12 = significance(maps['DR1'], 12)             # DR1 with a 12' kernel (annulus 36'-72'), shown in Figure 10.2
# colours clipped to -3 < S < 8 so that a few very large peaks do not compress the scale
hp.mollview(np.clip(S12, -3, 8), title='Gaia DR1: significance of stellar overdensities, 12′ Gaussian kernel', cmap='viridis',
            min=-3, max=8, unit='S')

**Figure 10.2.** Significance $S$ of stellar overdensities in the Gaia DR1 counts of Figure 10.1 (left), for a Gaussian
kernel of σ = 12′, with the mean and variance taken over the 36′–72′ annulus (3σ–6σ) and the variance floored at its
Poisson value; same projection as Figure 10.1, colour scale (viridis) clipped to −3 < S < 8. This is our reconstruction
of Figure 1 of Koposov et al. (2017). Compact yellow spots are known satellites and star clusters; the fine network of
filaments and arcs follows the DR1 scanning pattern, as described by Koposov et al. Dividing by the local scatter removes
the large-scale density gradients (the disc does not stand out), but the disc and the Magellanic Clouds are still full of
real small-scale structure (open clusters, dust) and are usually excluded from halo searches.

## 10.3 One kernel is not enough

A kernel matched to the object's size maximises its significance. We compute S with σ = 4′, 6′ and 12′ (our pixels are
3.4′; the paper used kernels from 1.5′ to 24′ on a finer grid) and look at four objects of different size: the two Gaia
discoveries, Boötes I and the very extended Crater 2.

In [ ]:
# Test objects: name, centre (ra, dec) [deg] and half-light radius r_h [arcmin], from 2' (Gaia 2) to 31' (Crater 2)
objs = pd.DataFrame([('Gaia 1', 101.47, -16.75, 6.5), ('Gaia 2', 28.12, 53.04, 1.9), ('Boötes I', 210.02, 14.51, 10.0),
                     ('Crater 2', 177.31, -18.41, 31.2)], columns=['name', 'ra', 'dec', 'rh_arcmin'])
SIGMAS = (4, 6, 12)             # kernel widths [arcmin]; 4' is the smallest we allow on 3.4' pixels (see 10.2)


def peak_at(S, ra, dec, radius_arcmin):
    """Largest S within radius of (ra, dec).

    S is a RING-ordered map (query_disc returns RING pixel indices by default); ra, dec in deg, radius in arcmin.
    """
    return S[hp.query_disc(NSIDE, hp.ang2vec(ra, dec, lonlat=True), np.radians(radius_arcmin / 60))].max()


Smaps = {s: (S12 if s == 12 else significance(maps['DR1'], s)) for s in SIGMAS}       # reuse the 12' map computed above
# maximum S within 5' (about 1.5 pixels) of each catalogued centre, per kernel: rows = objects, columns = kernels
tab = pd.DataFrame({f'σ = {s}′': [peak_at(Smaps[s], o.ra, o.dec, 5) for o in objs.itertuples()] for s in SIGMAS}, index=objs.name)
print('max S within 5′ of each object, Gaia DR1:')
print(tab.round(1).to_string())

# 3 x 4 grid of cut-outs: rows = kernels (4', 6', 12'), columns = objects. Each is 120 x 120 pixels of 1' = 2 x 2 deg,
# gnomonic projection centred on the object; same colour scale as the full-sky map (-3 < S < 8), no colour bar.
fig = plt.figure(figsize=(13, 10.5))                          # square panels: ~3.2 in wide, 3.5 in per row
for k, o in enumerate(objs.itertuples()):
    for j, s in enumerate(SIGMAS):
        hp.gnomview(Smaps[s], rot=(o.ra, o.dec), xsize=120, reso=1.0, fig=fig, sub=(len(SIGMAS), len(objs), j * len(objs) + k + 1),
                    min=-3, max=8, cmap='viridis', cbar=False, notext=True, title=f'{o.name}, σ = {s}′', margins=(0.01, 0.01, 0.01, 0.04))   # top margin for the title
# Gaia 1 must pass the 6-sigma threshold of Koposov et al. (2017) with the 4' kernel; Gaia 2 only S > 5, since our 3.4'
# pixels do not allow their 1.5' kernel; Crater 2 must appear (S > 5) with the 12' kernel matched to its size.
assert tab.loc['Gaia 1', 'σ = 4′'] > 6 and tab.loc['Gaia 2', 'σ = 4′'] > 5 and tab.loc['Crater 2', 'σ = 12′'] > 5

**Figure 10.3.** Gaia DR1 significance maps around Gaia 1, Gaia 2, Boötes I and Crater 2 (columns) for kernels of
σ = 4′, 6′ and 12′ (rows): 2° × 2° gnomonic cut-outs (120 pixels of 1′) centred on the catalogued positions, same colour
scale as Figure 10.2 (viridis, −3 < S < 8; no colour bar). The largest S within 5′ of each centre (table above, for
σ = 4′ / 6′ / 12′) is 6.6 / 5.9 / 2.7 for Gaia 1, 5.6 / 1.9 / −0.2 for Gaia 2, 6.4 / 6.0 / 4.3 for Boötes I and
0.4 / 1.7 / 5.9 for Crater 2. Gaia 1 ($r_h \approx 6.5′$, 10′ from Sirius) stands out with the 4′ and 6′ kernels and fades
at 12′; Gaia 2 ($r_h \approx 2′$) appears only with the smallest kernel, at S ≈ 5.6, below the paper's 6σ threshold,
because our 3.4′ pixels do not allow kernels as small as its 1.5′ ones; Crater 2 ($r_h \approx 31′$) appears only with
the largest. A search must scan sizes, and then calibrate for the many trials (Chapter 5).

## 10.4 DR1 versus DR3, and the value of parallaxes and colours

We now run the three kernels on the six maps (DR1; DR3; DR3 without stars of ϖ > 1 mas or > 0.2 mas, each with and
without the colour window) and ask two questions, away from the Galactic plane ($|b| > 20°$):

1. **Recovery**: how many known Milky Way satellites and star clusters (Local Volume Database v1.1.1, `host = mw`) have
   S > 6 within max(5′, $r_h$) of their position, in any kernel?
2. **Spurious peaks**: how many pixels have S > 6 farther than 1° from any known object, excluding the Magellanic Clouds
   (within 12° of the LMC and 6° of the SMC)? (A rough measure: real searches also mask bright galaxies and bright stars.)

In [ ]:
lv = load('lvdb_mw')                                    # LVDB v1.1.1: Milky Way satellites and clusters (host = mw)
rot = hp.Rotator(coord=['C', 'G'])                      # equatorial -> Galactic, to get latitudes b
pix_ra, pix_dec = hp.pix2ang(NSIDE, np.arange(NPIX), lonlat=True)    # centres of all RING pixels [deg], shape (NPIX,)
pix_b = rot(pix_ra, pix_dec, lonlat=True)[1]
lv_b = rot(lv.ra.to_numpy(), lv.dec.to_numpy(), lonlat=True)[1]
lv = lv[np.abs(lv_b) > 20].reset_index(drop=True)       # keep known objects away from the Galactic plane
# pixels within 1 deg of any known object: excluded from the count of spurious peaks
near_known = np.zeros(NPIX, bool)
for o in lv.itertuples():
    near_known[hp.query_disc(NSIDE, hp.ang2vec(o.ra, o.dec, lonlat=True), np.radians(1.0))] = True
pix_vec = np.array(hp.pix2vec(NSIDE, np.arange(NPIX)))  # unit vectors of the pixel centres, shape (3, NPIX)
# Magellanic Clouds: a pixel is within radius r of a centre if the dot product of their unit vectors exceeds cos(r);
# 12 deg around the LMC (ra, dec = 80.894, -69.756) and 6 deg around the SMC (13.187, -72.829)
clouds = (pix_vec.T @ hp.ang2vec(80.894, -69.756, lonlat=True) > np.cos(np.radians(12))) | \
         (pix_vec.T @ hp.ang2vec(13.187, -72.829, lonlat=True) > np.cos(np.radians(6)))
high_b = (np.abs(pix_b) > 20) & ~clouds                 # the area in which spurious peaks are counted

summary, recovered = [], {}
for name, H in maps.items():
    # per-pixel maximum of S over the three kernels (the DR1 maps were already computed above)
    Sk = np.array([Smaps[s] if name == 'DR1' else significance(H, s) for s in SIGMAS])   # (3, NPIX): one map per kernel
    Smax = Sk.max(0)
    # recovered = S > 6 within max(5', r_h) of the catalogued centre (r_h in arcmin; 5' when r_h is not catalogued)
    rec = np.array([peak_at(Smax, o.ra, o.dec, max(5, o.rhalf if np.isfinite(o.rhalf) else 5)) > 6 for o in lv.itertuples()])
    recovered[name] = rec
    # spurious = pixels with S > 6 at |b| > 20 deg, outside the Clouds, and more than 1 deg from any known object
    summary.append(dict(data=name, recovered=int(rec.sum()), of=len(lv),
                        spurious_pixels=int(np.sum((Smax > 6) & high_b & ~near_known))))
    if name == 'DR3, ϖ ≤ 1 + colour':
        S_best = Smax                                   # kept for the full-sky map below
        S_kern = Sk.argmax(0).astype(np.int8)           # index into SIGMAS of the best kernel, per pixel (Section 10.7)
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))

# Known objects in the plane of distance and absolute magnitude, with open markers for those recovered by each data set
fig, ax = plt.subplots(figsize=(7, 4))
mv = lv.M_V.to_numpy()
d = 10**(lv.distance_modulus.to_numpy() / 5 + 1) / 1e3   # heliocentric distance [kpc] from the distance modulus
for i, (name, mk) in enumerate(zip(maps, 'osD^vP')):     # one marker per data set (six)
    ax.scatter(d[recovered[name]], mv[recovered[name]], marker=mk, s=40, facecolors='none', edgecolors=f'C{i}', label=f'{name}: {recovered[name].sum()}')
ax.scatter(d, mv, s=4, c='k', label='all known, |b| > 20°')
ax.set(xscale='log', xlabel='distance [kpc]', ylabel='$M_V$', title='known objects recovered (S > 6)')
ax.invert_yaxis()                                        # brighter (more negative M_V) at the top
ax.legend(fontsize=8)
plt.tight_layout()
# full-sky map of max S over the kernels SIGMAS for the DR3 map with parallax and colour cuts; same scale as Figure 10.2
hp.mollview(np.clip(S_best, -3, 8), title='Gaia DR3, ϖ ≤ 1 mas + colour: max S over 4′, 6′, 12′ kernels', cmap='viridis', min=-3, max=8, unit='S')

**Figure 10.4.** Top: the 145 known Milky Way satellites and star clusters at $|b| > 20°$ (LVDB v1.1.1, `host = mw`;
black dots) in the plane of heliocentric distance (kpc, log scale, from the catalogued distance modulus) and absolute
magnitude $M_V$ (brighter upwards). The legend gives the number recovered (S > 6 within max(5′, $r_h$) of the catalogued
position, in any of the three kernels) by each data set: DR1 45, DR3 49, DR3 ϖ ≤ 1 50, DR3 ϖ ≤ 1 + colour 50,
DR3 ϖ ≤ 0.2 50, DR3 ϖ ≤ 0.2 + colour 51; open markers (one shape and colour per data set) mark the objects each
recovered. Bottom: the full-sky significance map of DR3 without stars of ϖ > 1 mas and with the colour window
(`n_plx1_cmd`), the per-pixel maximum of S over the σ = 4′, 6′ and 12′ kernels; same projection and colour scale as
Figure 10.2. It has 1597 pixels with S > 6 at $|b| > 20°$ more than 1° from any known Milky Way object (outside the
Magellanic Clouds), against 1214 for DR1, 1973 for DR3 without cuts, 1999 for DR3 ϖ ≤ 1, 2341 for DR3 ϖ ≤ 0.2 and 1788
for DR3 ϖ ≤ 0.2 + colour. The DR1 scanning arcs of Figure 10.2 are gone; the disc remains visible as a band of enhanced
small-scale fluctuations.

## 10.5 Removing nearby stars with parallaxes

At $17 < G < 21$ and high latitude, most stars belong to the Milky Way's disc and halo along the line of sight; a
satellite 20 kpc or more away has a parallax of 0.05 mas or less. Gaia DR3 measures parallaxes for most of these stars,
so we can drop stars whose parallax exceeds a threshold. We build two versions: without stars of **ϖ > 1 mas** (closer
than ~1 kpc, were the parallaxes exact) and without stars of **ϖ > 0.2 mas** (~5 kpc), each also with the colour window
$0.3 < G_{\rm BP} - G_{\rm RP} < 1.3$. The cuts use the measured parallax only, not its uncertainty, and stars without a
parallax are kept (they cannot be shown to be near).

The uncertainty matters for what the cuts do. It grows steeply with magnitude: in the Gaia cones of Section 10.7 (ten
high-latitude fields), the median $\sigma_\varpi$ is 0.10 mas at $G = 17$–18, 0.37 mas at 19–20, 0.68 mas at 20–20.5 and
1.19 mas at 20.5–21, where 43% of sources have no parallax at all (cf. ~0.5 mas at $G = 20$ in Lindegren et al. 2021,
https://ui.adsabs.harvard.edu/abs/2021A%26A...649A...2L). The measured parallax of a faint, distant star therefore
scatters far beyond 0.2 mas: the ϖ ≤ 0.2 cut removes nearby stars, but also a large, magnitude-dependent fraction of
distant ones, while the ϖ ≤ 1 cut removes few distant stars and few faint nearby ones. Figure 10.5 shows what each
cut keeps.

In [ ]:
hb = np.abs(pix_b) > 20                              # |b| > 20 deg
for k in maps:
    if k.startswith('DR3'):
        print(f'{k:22s}: {maps[k].sum():.3e} stars; fraction of DR3 kept at |b| > 20°: {maps[k][hb].sum() / maps["DR3"][hb].sum():.3f}')

# Figure 10.5, Galactic coordinates (centre l = 0, l increasing to the left). Top: log10 counts per pixel without stars of
# parallax > 1 mas and > 0.2 mas. Bottom: the fraction of the DR3 K17 stars each cut keeps, on coarser 55' pixels
# (nside 64; ud_grade with power=-2 sums the counts of the 256 small pixels in each), so that it is not dominated by noise
fig = plt.figure(figsize=(14, 9))
for n, k in enumerate(('DR3, ϖ ≤ 1', 'DR3, ϖ ≤ 0.2')):
    hp.mollview(np.log10(np.maximum(maps[k], 1)), coord=['C', 'G'], fig=fig, sub=(2, 2, n + 1), title=f'{k}: log10 counts per pixel',
                cmap='Greys', min=0, max=3, unit='', margins=(0.01, 0.06, 0.01, 0.06))
    n_k, n_all = hp.ud_grade(maps[k], 64, power=-2), hp.ud_grade(maps['DR3'], 64, power=-2)
    frac = np.where(n_all > 0, n_k / np.maximum(n_all, 1), hp.UNSEEN)
    hp.mollview(frac, coord=['C', 'G'], fig=fig, sub=(2, 2, n + 3), title=f'{k}: fraction of DR3 stars kept', cmap='viridis',
                min=0, max=1, unit='', margins=(0.01, 0.06, 0.01, 0.06))

**Figure 10.5.** Gaia DR3 (K17 selection, 17 < G < 21) after the plain parallax cuts, in Galactic coordinates
(Mollweide, $l = 0$ at the centre, increasing to the left). Top: $\log_{10}$ of the counts per 3.4′ pixel without stars of
ϖ > 1 mas (left; 1.37 × 10⁹ stars) and ϖ > 0.2 mas (right; 8.1 × 10⁸), grey scale 1 to 1000. Bottom: the fraction of the
DR3 stars that each cut keeps, on 55′ pixels (nside 64) so that the ratio is not dominated by noise; colour 0 to 1.
Over $|b| > 20°$ the cuts keep 79% and 41% of the stars. The fraction kept is highest where the stars are distant — the
bulge (~0.9 and 0.6–0.7) and the Magellanic Clouds (~1 and ~0.8) — and lowest at high latitude, where the nearby
disc and halo foreground dominates (~0.75–0.8 and ~0.35–0.45). Dark knots along the disc are dust clouds, behind
which only nearby stars are seen; faint arcs at high latitude follow Gaia's scanning pattern, where the parallax
quality — and hence the fraction removed — changes with the number of observations.

## 10.6 What does DR3 detect? From pixels to detections, and from detections to objects

Figure 10.4 counted *pixels* with $S > 6$. To ask which objects DR3 finds, we need *detections* — one per object — and a
way to decide whether each one is already known. Both steps can go wrong in instructive ways.

**Grouping pixels into detections.** We take every pixel with $S > 6$ in the map of DR3 without stars of ϖ > 1 mas,
with the colour window (the map of Figure 10.4), at $|b| > 20°$ and outside the Magellanic Clouds — the area in which
spurious pixels were counted. Pixels that touch (any of the 8 HEALPix neighbours) form one detection, placed at its
highest pixel and assigned the kernel that gave the maximum there. A simpler recipe — keep the highest pixel, discard
everything within a fixed radius, repeat — fails here: with a radius of 15′, bright globular clusters (NGC 5466, M2, M15
and others) stay above $S = 6$ beyond 15′ in the 12′ kernel, and each produced a ring of extra "detections" 13′–15′ from
its centre, all of which looked unmatched. Connected regions adapt to the size of each object.

**Matching detections to known objects.** A detection is matched if it lies within max(15′, 3 $r_h$) of an object of the
Local Volume Database — all 1727 objects of LVDB v1.1.1 (`lvdb_all`), not only the Milky Way satellites and clusters of
Section 10.4; the nearest such object wins, and objects without a catalogued $r_h$ get 15′. Two pitfalls:
1. *An incomplete catalogue.* The `host = mw` table of Section 10.4 leaves out globular clusters that LVDB assigns to
   the Sagittarius dwarf (NGC 2419, Palomar 12), clusters of the LMC (NGC 1841), M31, M33 and their satellites, and
   nearby galaxies such as NGC 253. Matched against it alone, many of the strongest detections look new.
2. *Chance coincidences.* Too generous a radius matches anything: discs of 1° around 1727 objects cover up to 13% of
   the sky (1727 × π deg² / 41,253 deg², ignoring overlaps). With max(15′, 3 $r_h$) the discs cover 2.1% of the searched
   area, so ~2 of the matches are expected by chance.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

# 1. Group: every S > 6 pixel of DR3 (parallax <= 1 mas, + colour) at |b| > 20 deg outside the Clouds; pixels that touch (any of the 8
# HEALPix neighbours) belong to the same detection; each detection is placed at its highest pixel.
sel = np.flatnonzero((S_best > 6) & high_b)
idx = -np.ones(NPIX, int); idx[sel] = np.arange(len(sel))         # position of each selected pixel in sel, or -1
nbr = hp.get_all_neighbours(NSIDE, sel)                           # (8, len(sel)) neighbour pixels (RING), -1 if none
a = np.repeat(np.arange(len(sel))[None], 8, 0).ravel(); b = idx[nbr.ravel()]
ok = (nbr.ravel() >= 0) & (b >= 0)                                # links between two selected pixels
n_reg, lab = connected_components(coo_matrix((np.ones(ok.sum()), (a[ok], b[ok])), shape=(len(sel),) * 2), directed=False)
peak = np.array([sel[lab == k][np.argmax(S_best[sel[lab == k]])] for k in range(n_reg)])   # highest pixel per region
det = pd.DataFrame(dict(pix=peak, S=S_best[peak], npix=np.bincount(lab), sigma=np.array(SIGMAS)[S_kern[peak]]))
det['ra'], det['dec'] = hp.pix2ang(NSIDE, det.pix.to_numpy(), lonlat=True)
det['l'], det['b'] = rot(det.ra.to_numpy(), det.dec.to_numpy(), lonlat=True)
det = det.sort_values('S', ascending=False, ignore_index=True)

# 2. Match: to every LVDB object (all hosts), within max(15', 3 r_h) of its centre; the nearest one wins
lva = load('lvdb_all'); lva = lva[np.isfinite(lva.ra)].reset_index(drop=True)
# kind of object, by LVDB table: Milky Way dwarfs and clusters (including those LVDB assigns to the LMC, SMC or the
# Sagittarius dwarf, and Milky Way candidates), clusters of other galaxies, and galaxies beyond the Milky Way
mw_like = lva.table.isin(['dwarf_mw', 'gc_harris', 'gc_mw_new', 'gc_ambiguous']) | \
          (lva.table.eq('candidate') & lva.host.isin(['mw', 'lmc', 'smc', 'sagittarius_1']))
lva['kind'] = np.where(mw_like, 'Milky Way satellite or cluster',
                       np.where(lva.table.isin(['gc_dwarf_hosted', 'gc_other']), 'cluster of another galaxy',
                                'galaxy beyond the Milky Way'))
r_match = np.maximum(15, 3 * np.nan_to_num(lva.rhalf.to_numpy())) / 60      # match radius [deg]; r_h unknown -> 15'
ang = np.degrees(np.arccos(np.clip(hp.ang2vec(det.ra.to_numpy(), det.dec.to_numpy(), lonlat=True)
                                   @ hp.ang2vec(lva.ra.to_numpy(), lva.dec.to_numpy(), lonlat=True).T, -1, 1)))  # (n_det, n_lvdb)
hit = ang < r_match[None]
best = np.where(hit, ang, np.inf).argmin(1)
det['match'] = np.where(hit.any(1), lva.name.to_numpy()[best], '')
det['kind'] = np.where(hit.any(1), lva.kind.to_numpy()[best], 'unmatched')
# chance coincidences: the fraction of the searched sky that lies within a match radius of some LVDB object
covered = np.zeros(NPIX, bool)
for o, r in zip(lva.itertuples(), r_match):
    covered[hp.query_disc(NSIDE, hp.ang2vec(o.ra, o.dec, lonlat=True), np.radians(r))] = True
f_cov = covered[high_b].mean()
print(f'{len(sel)} pixels with S > 6 -> {n_reg} detections; matched {(det.match != "").sum()}, unmatched {(det.match == "").sum()}')
print(det.kind.value_counts().to_string())
# the same matching against the host = mw objects alone (the table of Section 10.4, plus its |b| < 20 deg objects)
print(f'matched to host = mw objects alone: {(hit & lva.host.eq("mw").to_numpy()[None]).any(1).sum()}')
print('galaxies matched:', ', '.join(det.match[det.kind == 'galaxy beyond the Milky Way']))
print(f'match discs cover {f_cov:.4f} of the searched sky: ~{f_cov * n_reg:.1f} chance matches expected')
unm = det[det.match == ''].reset_index(drop=True)
print(unm[['S', 'npix', 'sigma', 'ra', 'dec', 'l', 'b']].round(2).to_string())

# Figure 10.6: the detections in Galactic coordinates (Mollweide; l increasing to the left), marker area ~ S
fig = plt.figure(figsize=(13, 6.5)); ax = fig.add_subplot(projection='mollweide')
lrad = lambda l: -np.radians(((l + 180) % 360) - 180)             # l = 0 at the centre, increasing to the left
for kind, c in (('Milky Way satellite or cluster', 'C0'), ('cluster of another galaxy', 'C2'),
                ('galaxy beyond the Milky Way', 'C1'), ('unmatched', 'C3')):
    d = det[det.kind == kind]
    ax.scatter(lrad(d.l), np.radians(d.b), s=8 * np.clip(d.S, 6, 60), c=c, alpha=0.7, label=f'{kind}: {len(d)}')
for k, u in unm.iterrows():
    ax.annotate(f'#{k}', (lrad(u.l), np.radians(u.b)), fontsize=8, xytext=(4, 4 if k % 2 else -11), textcoords='offset points')
for bb in (-20, 20):
    ax.plot(np.linspace(-np.pi, np.pi, 200), np.full(200, np.radians(bb)), c='0.5', lw=0.8, ls='--')
ax.set_xticks(np.radians([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150]))
ax.set_xticklabels(['150°', '120°', '90°', '60°', '30°', '0°', '330°', '300°', '270°', '240°', '210°'])
ax.grid(alpha=0.3); ax.legend(loc='lower right', fontsize=8, markerscale=0.5)
ax.set_title('Gaia DR3, ϖ ≤ 1 mas + colour: detections with S > 6 at |b| > 20° (marker area ∝ S)')
plt.tight_layout()

# Tests: the unmatched detections are the positions around which gaia_dr3_ch10_cands was extracted (same analysis),
# and most detections are known objects
centres = np.array([[float(v) for v in re.search(r'q3c_radial_query\(ra, dec, ([-\d.]+), ([-\d.]+), 1\.0\)', q).groups()]
                    for q in MANIFEST['gaia_dr3_ch10_cands2']['queries']])
assert len(centres) == len(unm) and np.allclose(centres, unm[['ra', 'dec']].to_numpy(), atol=1e-3)
assert (det.match != '').mean() > 0.85

**Figure 10.6.** The 106 detections (connected regions of $S > 6$) of the DR3 ϖ ≤ 1 + colour map at $|b| > 20°$ outside
the Magellanic Clouds, in Galactic coordinates (Mollweide projection, $l = 0$ at the centre and increasing to the left;
dashed lines: $b = \pm 20°$). Marker area grows with $S$ (clipped at 60). Colour: the kind of LVDB object matched within
max(15′, 3 $r_h$), by LVDB table — blue, a Milky Way satellite or cluster (55, including those LVDB assigns to the LMC
or the Sagittarius dwarf); green, a cluster of another galaxy (4); orange, a galaxy
beyond the Milky Way (36: M33, Local Group dwarfs such as IC 1613, WLM and Leo A, and Local Volume galaxies such as
NGC 253, M81 and M101); red, unmatched (11, numbered #0–#10 in decreasing $S$). The empty band around $b = 0$ and
the empty caps around the Clouds ($l \approx 280°$–$300°$, $b \approx -33°$ to $-44°$) were not searched. Detections crowd
towards the north Galactic cap ($b > 60°$), where the Virgo cluster of galaxies lies; #0, #1, #3 and #7 are there.

**From 5238 pixels to 106 detections to 11 unknowns.** Grouping reduces the 5238 pixels with $S > 6$ to 106 detections:
most bright objects cover tens of pixels. Matching to the complete LVDB explains 95 of them; ~2 of these matches are
expected by chance. Matched to the `host = mw` objects of LVDB alone, only 50 detections would have been explained and
56 would have looked new.

## 10.7 Report cards of the unmatched detections

The unmatched detections get report cards, the Gaia version of those of Chapter 9. They use `gaia_dr3_ch10_cands2`: all
Gaia DR3 sources with $17 < G < 21$ within 1° of each unmatched detection, *without* the excess-noise cut, so that
extended sources can be shown too.

**Top row: maps of 30′ × 30′ or more** (east to the left; the blue circle is the aperture, of radius twice the kernel
that gave the detection):
1. sources passing the Koposov et al. (2017) cut (K17: $17 < G < 21$ and the excess-noise limit);
2. the selection behind the map: K17, ϖ ≤ 1 mas or no parallax, $0.3 <$ BP − RP $< 1.3$;
3. $S$ (maximum over the three kernels);
4. sources *failing* the excess-noise cut — Gaia's indicator of extended sources: galaxies, and blends in crowded fields.

**Bottom row:** G vs BP − RP Hess diagrams of K17 sources in the aperture, in the 30′–60′ annulus, and their difference
with the annulus scaled by the area ratio α (dashed lines: the colour window); and the proper motions of the selected
sources in the aperture (red) and the annulus (grey), with the weighted mean parallax of each in the title.

**What to look for.**
- *A Milky Way satellite:* a concentration of selected sources with an excess along one giant branch or horizontal
  branch in the Hess difference, and a clump in proper motion away from the foreground's; its stars have parallaxes
  consistent with zero.
- *A galaxy beyond the Local Group:* a concentration of extended sources (panel 4) at the position; the excess is made of
  compact sources the galaxy contains or that Gaia splits from it — globular clusters (red, BP − RP ≈ 1–1.3, $G > 20$),
  star-forming knots (blue); their proper motions scatter around zero, so the median proper motion of the aperture is
  smaller than the annulus's. Their parallaxes are not a reliable test: Gaia's parallaxes of knots and blends in a galaxy
  can be biased high (NGC 4490 below).
- *Dust:* a patchy excess at the red edge of the colour window, where extinction moves stars into or out of the
  selection, with no proper-motion or parallax signature.

In [ ]:
def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


def untangent(xi, eta, ra0, dec0):
    """Inverse gnomonic projection: (xi, eta) [deg] -> (ra, dec) [deg]."""
    xi, eta, ra0, dec0 = np.radians(xi), np.radians(eta), np.radians(ra0), np.radians(dec0)
    rho = np.hypot(xi, eta); c = np.arctan(rho)
    dec = np.arcsin(np.cos(c) * np.sin(dec0) + np.where(rho > 0, eta * np.sin(c) * np.cos(dec0) / np.maximum(rho, 1e-300), 0))
    ra = ra0 + np.arctan2(xi * np.sin(c), rho * np.cos(dec0) * np.cos(c) - eta * np.sin(dec0) * np.sin(c))
    return np.degrees(ra) % 360, np.degrees(dec)


def gaia_card(T, ra0, dec0, sig, label, r_off=(30, 60)):
    """Report card of a detection at (ra0, dec0) [deg] found with kernel sig [arcmin], from Gaia sources T (1 deg cone).

    Top: maps around the detection (east to the left; blue circle: aperture of radius 2 sig) of sources passing the
    Koposov et al. (2017) cut, of the selection behind the map (ϖ ≤ 1 + colour), of S, and of sources failing the
    excess-noise cut (mostly galaxies). Bottom: G vs BP-RP Hess diagrams of the aperture, of the r_off annulus and their
    area-scaled difference, and proper motions of the ϖ ≤ 1 + colour sources in the aperture and annulus.
    """
    x, y = (np.array(v) * 60 for v in tangent(T.ra.to_numpy(), T.dec.to_numpy(), ra0, dec0))   # offsets [arcmin]
    G, c = T.phot_g_mean_mag.to_numpy(), T.bp_rp.to_numpy()
    k17 = T.astrometric_excess_noise.to_numpy() < 10**(0.15 * (G - 15) + 0.25)   # the Koposov et al. (2017) cut
    near = (T.parallax > 1).fillna(False).to_numpy()                             # parallax > 1 mas (no parallax: kept)
    use = k17 & ~near & (c > 0.3) & (c < 1.3)                                    # the selection of the map
    r_on = 2 * sig; half = max(15, 1.25 * r_on)                                  # aperture radius, map half-size [arcmin]
    r = np.hypot(x, y); on, off = r < r_on, (r >= r_off[0]) & (r < r_off[1])
    alpha = r_on**2 / (r_off[1]**2 - r_off[0]**2)                               # area ratio aperture / annulus
    fig = plt.figure(figsize=(18, 8.4)); gs = fig.add_gridspec(2, 4, hspace=0.38, wspace=0.5)
    ax = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(4)]
    e = np.arange(-half, half + 0.01, 1.0)                                        # 1' map pixels
    dens = lambda s: gaussian_filter(np.histogram2d(x[s], y[s], [e, e])[0], 1.5)  # per arcmin^2, 1.5' smoothing
    gx, gy = np.meshgrid(0.5 * (e[1:] + e[:-1]), 0.5 * (e[1:] + e[:-1]), indexing='ij')
    Smap = S_best[hp.ang2pix(NSIDE, *untangent(gx / 60, gy / 60, ra0, dec0), lonlat=True)]   # S at the map pixels
    for a, (img, ttl, cm, lim) in zip(ax[:4], ((dens(k17), 'K17 sources [arcmin⁻²]', 'Greys', None),
                                              (dens(use), 'ϖ ≤ 1 + colour selection [arcmin⁻²]', 'Greys', None),
                                              (Smap, 'S (max over kernels)', 'viridis', (-3, 8)),
                                              (dens(~k17), 'failing the excess-noise cut [arcmin⁻²]', 'Greys', None))):
        im = a.imshow(img.T, origin='lower', extent=(-half, half, -half, half), cmap=cm, vmin=lim and lim[0], vmax=lim and lim[1])
        a.add_patch(plt.Circle((0, 0), r_on, fill=False, color='C0', lw=1))
        a.set(xlim=(half, -half), ylim=(-half, half), title=ttl, xlabel='Δξ [′]')
        plt.colorbar(im, ax=a, fraction=0.046, pad=0.04)
    ax[0].set_ylabel('Δη [′]')
    hc, hm = np.arange(-0.5, 2.51, 0.1), np.arange(17, 21.01, 0.25)
    H = lambda s: np.histogram2d(c[s], G[s], [hc, hm])[0]
    D = H(on & k17) - alpha * H(off & k17)
    vd = max(np.abs(D).max(), 1)
    for a, (img, ttl, cm, v0, v1) in zip(ax[4:7], ((H(on & k17), f'aperture R < {r_on:.0f}′: {(on & k17).sum()} K17 sources', 'Greys', 0, None),
                                                  (H(off & k17), f'annulus {r_off[0]}′–{r_off[1]}′: {(off & k17).sum()}', 'Greys', 0, None),
                                                  (D, f'aperture − α × annulus (α = {alpha:.3f})', 'RdBu_r', -vd, vd))):
        im = a.imshow(img.T, origin='lower', extent=(hc[0], hc[-1], hm[0], hm[-1]), aspect='auto', cmap=cm, vmin=v0, vmax=v1)
        for cc_ in (0.3, 1.3):
            a.axvline(cc_, c='C1', lw=0.8, ls='--')                              # the colour window of the selection
        a.set(xlim=(-0.5, 2.5), ylim=(21, 17), xlabel='BP − RP', title=ttl)
        plt.colorbar(im, ax=a, label='sources per bin')
    ax[4].set_ylabel('G')
    # proper motions of the ϖ ≤ 1 + colour sources: annulus (grey) and aperture (red); extragalactic sources sit at (0, 0)
    pm = np.isfinite(T.pmra.to_numpy())
    ax[7].scatter(T.pmra[use & off & pm], T.pmdec[use & off & pm], s=1, c='0.7', label='annulus')
    ax[7].scatter(T.pmra[use & on & pm], T.pmdec[use & on & pm], s=8, c='C3', label='aperture')
    ax[7].axhline(0, c='k', lw=0.5); ax[7].axvline(0, c='k', lw=0.5)
    wm = lambda s: (np.average(T.parallax[s], weights=T.parallax_error[s]**-2), np.sum(T.parallax_error[s]**-2)**-0.5)
    (p_on, e_on), (p_off, e_off) = wm(use & on & pm), wm(use & off & pm)
    ax[7].set(xlim=(-10, 10), ylim=(-10, 10), aspect='equal', xlabel='μ_α* [mas/yr]', ylabel='μ_δ [mas/yr]',
              title=f'ϖ: aperture {p_on:.3f} ± {e_on:.3f}, annulus {p_off:.3f} ± {e_off:.3f} mas')
    ax[7].legend(fontsize=7, loc='lower left', markerscale=3)
    n_on, n_off = (use & on).sum(), (use & off).sum()
    fig.suptitle(f'{label}: S = {S_best[hp.ang2pix(NSIDE, ra0, dec0, lonlat=True)]:.1f}, σ = {sig}′; ϖ ≤ 1 + colour sources in '
                 f'the aperture {n_on} vs {alpha * n_off:.1f} expected (excess {n_on - alpha * n_off:.1f} ± {np.sqrt(n_on + alpha**2 * n_off):.1f})')
    plt.show()
    # numbers for the summary table: excess, parallaxes, median proper motion of aperture and annulus (ϖ ≤ 1 + colour
    # sources), and the excess of sources failing the excess-noise cut (extended sources) in the aperture
    pmm = lambda s: np.hypot(np.median(T.pmra[s]), np.median(T.pmdec[s]))
    n_ext_on, n_ext_off = (on & ~k17).sum(), (off & ~k17).sum()
    return dict(excess=n_on - alpha * n_off, err=np.sqrt(n_on + alpha**2 * n_off), plx_on=p_on, plx_on_err=e_on,
                plx_off=p_off, pm_on=pmm(use & on & pm), pm_off=pmm(use & off & pm),
                ext_excess=n_ext_on - alpha * n_ext_off, ext_err=np.sqrt(n_ext_on + alpha**2 * n_ext_off))


cones = load('gaia_dr3_ch10_cands2')                  # Gaia DR3, 17 < G < 21, within 1 deg of each unmatched detection
card_res = pd.DataFrame({k: gaia_card(cones[cones.cand == k], u.ra, u.dec, u.sigma,
                                      f'#{k} (ra {u.ra:.2f}, dec {u.dec:.2f}; l {u.l:.1f}, b {u.b:.1f})')
                         for k, u in unm.iterrows()}).T
# summary: S and kernel, excess of ϖ ≤ 1 + colour sources, weighted mean parallax in the aperture and the annulus [mas],
# |median proper motion| in the aperture and the annulus [mas/yr], excess of sources failing the excess-noise cut
print(pd.concat([unm[['S', 'sigma']], card_res], axis=1).round(2).to_string())

**Figure 10.7.** Report cards of the 11 unmatched detections, #0–#10 in decreasing $S$, as described at the start of
this section; the aperture radius is twice the kernel σ of the detection (8′ for σ = 4′, 12′ for σ = 6′) and the annulus
is 30′–60′. The table above the cards' figures lists, for each detection: $S$ and σ; the excess of selected sources
(ϖ ≤ 1 mas or no parallax, 0.3 < BP − RP < 1.3) in the aperture over α × annulus, with its Poisson error; the weighted
mean parallax of these sources in the aperture and in the annulus (mas); the amplitude of their median proper motion in
the aperture and in the annulus (mas yr⁻¹); and the excess of sources *failing* the excess-noise cut (extended sources).

**What the unmatched detections are.** We looked up each position in SIMBAD (https://simbad.cds.unistra.fr; galaxies
within 15′, largest first; done by the authors with the copy of SIMBAD in WSDB):

| # | S | SIMBAD | type | card |
|---|---|---|---|---|
| 0 | 14.1 | M87, 1.3′ (centre of the Virgo cluster) | giant elliptical | extended excess 10.6 ± 4.1; median PM 1.8 vs 4.3 mas/yr; ϖ 0.11 vs 0.28 mas; red compact excess (BP − RP ≈ 1.1–1.2, G > 20): globular clusters |
| 1 | 11.4 | NGC 4490, 1.0′ (with NGC 4485) | interacting galaxies | extended excess 37.9 ± 6.5; PM 3.3 vs 5.0; ϖ 0.47 vs 0.30 (biased knots) |
| 2 | 10.1 | M77, 1.5′ | Seyfert 2 galaxy | extended excess 34.6 ± 6.1; PM 2.2 vs 3.7 |
| 3 | 10.1 | NGC 4636, 1.7′ | elliptical (Virgo) | extended excess 8.5 ± 3.6; PM 1.9 vs 4.3 |
| 4 | 9.8 | NGC 3310, 1.7′ | starburst galaxy | extended excess 34.0 ± 6.3; PM 3.4 vs 4.9 |
| 5 | 7.3 | NGC 1399, 2.0′ (centre of the Fornax cluster) | giant elliptical | extended excess 14.7 ± 5.0; PM 3.1 vs 4.0 |
| 6 | 7.3 | LDN 1452, 12′ (Perseus) | dark cloud | no extended excess (1.8 ± 2.5); PM 1.0 vs 1.1; excess at the red edge of the colour window |
| 7 | 7.3 | NGC 4559, 1.1′ | spiral (HII galaxy) | extended excess 8.3 ± 3.5; PM 3.7 vs 4.9 |
| 8 | 6.5 | M31's outer disc (HII regions within 9′–15′; centre 0.8° away) | Local Group galaxy | excess 1084 ± 37; PM 0.2 vs 1.3; ϖ 0.10 vs 0.28 |
| 9 | 6.3 | none (a galaxy-cluster candidate 9′ away) | — | no extended excess; PM 3.4 vs 3.2; ϖ 0.33 vs 0.34 |
| 10 | 6.0 | none | — | no extended excess; PM 3.5 vs 3.1; ϖ 0.33 vs 0.32 |

*Reading.* Seven of the eleven are galaxies beyond the Local Volume — the giant ellipticals at the centres of the Virgo
and Fornax clusters with their globular-cluster systems, and nearby star-forming galaxies — that LVDB, a catalogue of
dwarfs and of galaxies within ~10 Mpc, does not list. Their cards agree: an excess of extended sources at the position,
and selected sources whose median proper motion is smaller than the field's, as for extragalactic objects. #8 is M31
itself: LVDB gives it no $r_h$, so the 15′ match radius missed a detection 0.8° from its centre, in its outer disc —
a reminder to give large objects large radii. #6 is dust: the edge of the dark cloud LDN 1452, where extinction moves
stars across the red edge of the colour window, with no proper-motion, parallax or extended-source signature. #9 and
#10, at the threshold ($S$ = 6.3 and 6.0), show no distinct population at all: their selected sources have the
proper motions and parallaxes of the annulus. They are most likely fluctuations of the field — or structures too
diffuse to characterise with Gaia alone; deeper photometry would decide. No new Milky Way satellite is found at
$S > 6$ — as expected, since Gaia's depth was searched by Koposov et al. (2017) and many surveys since.

## Discussion

- **DR3 versus DR1.** With the same selection, DR3 recovers somewhat more known objects than DR1 (49 against 45 of 145
  at $|b| > 20°$): more stars per object, from deeper completeness at G ≈ 20–21 after 34 instead of 14 months. It also has
  *more* S > 6 pixels away from catalogued Milky Way objects (1973 against 1214). Section 10.6 shows what they are: grouped
  into detections and matched to the whole of LVDB, 95 of 106 detections are known objects — many of them galaxies beyond
  the Milky Way — and of the other 11, seven are more distant galaxies, one is M31's disc and one a dark cloud.
- **Parallax cuts.** Without stars of ϖ > 1 mas, 79% of the K17 stars at $|b| > 20°$ remain; without ϖ > 0.2 mas, 41%.
  Both recover one more known object than plain DR3 (50 against 49), but neither reduces the spurious pixels on its own
  (1999 and 2341 against 1973). At G > 20 a parallax cut removes stars almost at random (Section 10.5), so it lowers
  the counts — and the significance of real objects — without much improving the contrast; and where the parallax
  quality varies across the sky, so does the fraction removed (Figure 10.5, bottom).
- **Colour cut.** Keeping 0.3 < BP − RP < 1.3 recovers as many known objects with fewer spurious pixels (1597 against
  1999 with ϖ ≤ 1; 1788 against 2341 with ϖ ≤ 0.2); it favours old, metal-poor populations (Boötes I and Crater 2 become
  more significant) but suppresses objects with other colours — Gaia 1 and Gaia 2 lose most of their signal — and drops
  stars without BP/RP photometry. A filter matched to one population is blind to others: the same trade-off as the
  isochrone filters of Chapters 4–5.
- **Known is not the same as "in the catalogue at hand".** Matching against an incomplete list makes known objects look
  new; matching with too generous a radius makes new objects look known. State the catalogue, its scope and the radius,
  and estimate the chance-coincidence rate (Section 10.6).

## Summary

- Database-side HEALPix histograms make all-sky density maps of 10⁹ stars cheap.
- The Koposov et al. (2017) statistic — kernel density minus its annulus mean, over the annulus scatter — reproduces their
  Figure 1 from Gaia DR1.
- Kernel size matters: compact clusters and extended dwarfs need different kernels.
- DR3 and its parallaxes and colours change both the recovery of known objects and the number of spurious peaks; plain
  parallax cuts help little at G > 20, where parallax errors exceed 0.5 mas.
- Pixels must be grouped into detections and matched to a complete catalogue: of 106 detections at $|b| > 20°$, 95 are
  known objects; report cards show that the 11 others are galaxies beyond the Local Volume, M31's disc, a dark cloud and
  two threshold fluctuations — no new satellite.